# Laboratorio 13 — El jugador aprende a cruzar el lago
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 18 de noviembre de 2026 · Clase 13 · Lab. Central 114
**Horario del lab:** ~11:50–13:35 (105 min)
**Viene de:** deck «Clase 13 - Aprendizaje por refuerzo»: diapo 3 (el bucle agente–entorno), diapo 4 (política, episodio, retorno), diapo 5 (el futuro depende solo del estado actual), diapo 8 (explorar o explotar), diapo 9 (la tabla Q), diapo 10 (cuando la tabla no alcanza: una red mira la pantalla) y diapo 13 (diseño de la recompensa).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **Apertura** | Nadie le enseña el camino | 5 min |
| **1** | Conocer el lago | 15 min |
| **2** | La tabla del jugador | 25 min |
| **3** | El hielo resbala | 15 min |
| **4** | El lago grande (8 × 8) | 20 min |
| **5** | Explorar y diseñar los puntos | 10 min |
| **6** | Cuando la tabla no alcanza: Mario | 5 min |
| **7** | Reflexión | 5 min |
| — | Margen | 5 min |
| **Entregable** | Respuestas y tus valores del lago grande | al final |

## Objetivos de aprendizaje

1. Ver con tus propios ojos que un jugador que parte moviéndose al azar aprende solo, partida tras partida, a cruzar un lago congelado sin caer en los hoyos.
2. Entender **la tabla del jugador** (la tabla Q de la diapo 9): una fila por casilla, una columna por movimiento, y cómo se va llenando desde la meta hacia atrás.
3. Ver qué cambia cuando el hielo resbala y cuando el lago es más grande, y elegir tú los valores para que el jugador cruce el lago grande.
4. Ver por qué conviene probar cosas nuevas, cómo los puntos deciden qué aprende y por qué en Mario ya no alcanza una tabla.

> **Qué necesitas.** Solo `gymnasium`, `numpy`, `pandas` y `matplotlib` (vienen en `requirements.txt`). Funciona en tu computador o en Colab, con cualquier Python reciente, y todo corre en segundos en la CPU.

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "gymnasium>=1.0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import HTML, Markdown, Video, display

SEED = 42
RAPIDO = False   # True = menos semillas en los experimentos, solo para ensayar el código
N_SEMILLAS = 2 if RAPIDO else 5   # cuántas veces repetimos un experimento, con distinta suerte
T0 = time.time()
random.seed(SEED)
np.random.seed(SEED)

# Paleta del deck «Clase 13 - Aprendizaje por refuerzo»
AZUL, NARANJA, MARINO, GRIS, GRIS_CLARO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#d9dbdf"
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
print("✔ Listo · Python", sys.version.split()[0], "| gymnasium", gym.__version__, "| RAPIDO =", RAPIDO,
      "| Colab =", IN_COLAB)

### Herramientas (ya escritas)

Estas funciones se entregan hechas: dibujar el lago, animar una partida, medir qué tan seguido llega el jugador a la meta y anotar en la tabla. Ejecuta la celda; no hace falta modificarla.

In [ ]:
MOVIMIENTOS = ["izquierda", "abajo", "derecha", "arriba"]   # el orden que usa FrozenLake: 0, 1, 2, 3
FLECHAS = ["←", "↓", "→", "↑"]
COLOR_CASILLA = {"S": GRIS_CLARO, "F": "#fffefb", "H": MARINO, "G": NARANJA}


def crear_lago(tamano="4x4", resbala=False):
    """El lago congelado de gymnasium: 4x4 (16 casillas) u 8x8 (64), con o sin hielo resbaloso.
    Cada partida se corta a los 100 instantes en el lago chico y a los 200 en el grande."""
    return gym.make("FrozenLake-v1", map_name=tamano, is_slippery=resbala,
                    max_episode_steps=100 if tamano == "4x4" else 200)


def pct(x, decimales=1):
    """Un porcentaje con coma decimal: 0.012 -> '1,2 %'."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")


def mapa(env):
    """El lago como una grilla de letras: S inicio, F hielo, H hoyo, G meta."""
    return np.array([[c.decode() for c in fila] for fila in env.unwrapped.desc])


def dibujar_lago(env, Q=None, camino=None, ax=None, titulo="", numeros=False, valores=False):
    """Dibuja el lago. Con Q: una flecha por casilla (su mejor movimiento según la tabla; · si aún no sabe).
    Con valores=True, el azul es más fuerte donde la tabla tiene números más altos."""
    m = mapa(env)
    n = m.shape[0]
    ax = ax or plt.gca()
    tope = Q.max() if Q is not None and Q.max() > 0 else 1.0
    for f in range(n):
        for c in range(n):
            casilla, letra = f * n + c, m[f, c]
            fondo = COLOR_CASILLA[letra]
            if valores and Q is not None and letra in "SF" and Q[casilla].max() > 0:
                fondo = plt.matplotlib.colors.to_rgba(AZUL, 0.15 + 0.75 * Q[casilla].max() / tope)
            ax.add_patch(Rectangle((c, n - 1 - f), 1, 1, fc=fondo, ec=GRIS, lw=0.8))
            centro = (c + 0.5, n - 1 - f + 0.5)
            tam = 18 if n == 4 else 10
            if letra in "HG":
                ax.text(*centro, "hoyo" if letra == "H" else "meta", ha="center", va="center", color="white",
                        fontsize=tam * 0.55, fontweight="bold")
            elif Q is not None:
                txt = FLECHAS[int(Q[casilla].argmax())] if Q[casilla].max() > 0 else "·"
                ax.text(*centro, txt, ha="center", va="center", fontsize=tam, color=MARINO)
            elif letra == "S":
                ax.text(*centro, "inicio", ha="center", va="center", fontsize=tam * 0.55, color=MARINO)
            if numeros:
                ax.text(c + 0.06, n - 1 - f + 0.08, str(casilla), fontsize=tam * 0.45, color=GRIS)
    if camino is not None and len(camino) > 1:
        xs = [s % n + 0.5 for s in camino]
        ys = [n - 1 - s // n + 0.5 for s in camino]
        ax.plot(xs, ys, color=AZUL, lw=2.5, alpha=0.8, marker="o", ms=4)
    ax.set(xlim=(0, n), ylim=(0, n), aspect="equal", title=titulo)
    ax.axis("off")
    return ax


def jugar(env, Q=None, seed=0, max_instantes=None):
    """Juega una partida. Q=None: movimientos al azar; si no, siempre el mejor movimiento de la tabla.
    Devuelve el camino (casillas visitadas), si llegó a la meta y cuántos instantes duró."""
    rng = np.random.default_rng(seed)
    casilla, _ = env.reset(seed=seed)
    camino, terminada, cortada, puntos_total = [casilla], False, False, 0.0
    while not (terminada or cortada):
        mov = int(rng.integers(4)) if Q is None else int(Q[casilla].argmax())
        casilla, puntos, terminada, cortada, _ = env.step(mov)
        camino.append(casilla)
        puntos_total += puntos
        if max_instantes and len(camino) > max_instantes:
            break
    return {"camino": camino, "llego": puntos_total > 0 and mapa(env).flatten()[casilla] == "G",
            "instantes": len(camino) - 1, "puntos": puntos_total}


def medir(env, Q=None, n=1000, seed=0):
    """Proporción de partidas (de n) en que el jugador llega a la meta. Q=None: al azar."""
    return float(np.mean([jugar(env, Q, seed=seed + i)["llego"] for i in range(n)]))


def mejor_posible(env):
    """Qué tan seguido llegaría a la meta el MEJOR jugador posible. Lo calculamos con el mapa completo
    del lago y sabiendo exactamente cómo resbala (el jugador nunca ve esa información)."""
    P, n_casillas, limite = env.unwrapped.P, env.observation_space.n, env.spec.max_episode_steps
    v = np.zeros(n_casillas)
    for _ in range(limite):
        v = np.array([max(sum(p * (r + (0 if fin else v[s2])) for p, s2, r, fin in P[s][a]) for a in range(4))
                      for s in range(n_casillas)])
    return float(v[0])


def animar(env, camino, titulo="", max_cuadros=60):
    """Anima una partida: el círculo naranja es el jugador."""
    camino = camino[:max_cuadros]
    n = mapa(env).shape[0]
    fig, ax = plt.subplots(figsize=(3.6, 3.6) if n == 4 else (4.4, 4.4), dpi=72)
    dibujar_lago(env, ax=ax, titulo=titulo)
    (punto,) = ax.plot([], [], "o", color=NARANJA, ms=22 if n == 4 else 12, mec=MARINO, mew=1.5)
    rastro, = ax.plot([], [], color=NARANJA, lw=2, alpha=0.5)

    def cuadro(k):
        xs = [s % n + 0.5 for s in camino[:k + 1]]
        ys = [n - 1 - s // n + 0.5 for s in camino[:k + 1]]
        punto.set_data([xs[-1]], [ys[-1]])
        rastro.set_data(xs, ys)
        return punto, rastro

    anim = FuncAnimation(fig, cuadro, frames=len(camino), interval=350, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode="once"))


def curiosidad_que_baja(partida, n_partidas, inicio=1.0, fin=0.05):
    """La curiosidad parte en 1 (todo al azar) y baja hasta 0,05 en el 80 % de las partidas."""
    return inicio * (fin / inicio) ** min(partida / (0.8 * n_partidas), 1.0)


def elegir(Q, casilla, curiosidad, rng):
    """Con probabilidad `curiosidad`, un movimiento al azar (explorar).
    Si no, el mejor movimiento según la tabla (explotar); si hay empate, uno de los empatados al azar."""
    if rng.random() < curiosidad:
        return int(rng.integers(Q.shape[1]))
    mejores = np.flatnonzero(Q[casilla] == Q[casilla].max())
    return int(rng.choice(mejores))


def anotar(Q, casilla, mov, puntos, casilla_nueva, terminada, aprendizaje, paciencia):
    """Anota en la tabla lo que pasó al hacer `mov` en `casilla`.
    Lo que vale ese movimiento = los puntos que dio + lo bueno que parece la casilla a la que llegó
    (si la partida terminó, ya no hay casilla siguiente). `paciencia` dice cuánto pesa ese futuro.
    El número de la tabla se acerca a ese valor: un poco si `aprendizaje` es chico, mucho si es grande."""
    lo_que_vale = puntos + (0.0 if terminada else paciencia * Q[casilla_nueva].max())
    Q[casilla, mov] += aprendizaje * (lo_que_vale - Q[casilla, mov])


print("✔ Herramientas listas")

---
## Apertura — Nadie le enseña el camino (5 min)

> **En palabras simples (diapo 3).** Hoy un jugador tiene que cruzar un lago congelado, desde el inicio hasta la meta, sin caer en los hoyos. Nadie le dice el camino. Solo le damos **1 punto si llega a la meta** (y 0 en cualquier otro caso). El jugador prueba, se cae miles de veces y de a poco descubre qué le conviene hacer en cada casilla.

Es el bucle de la diapo 3, con los nombres del lago. En los libros, el jugador se llama **agente**; aquí le diremos **el jugador**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.set(xlim=(0, 10), ylim=(-0.2, 3.1))
ax.axis("off")


def caja(x, y, titulo, detalle, fondo, texto="white"):
    ax.add_patch(FancyBboxPatch((x, y), 2.6, 1.0, boxstyle="round,pad=0.04,rounding_size=0.15", fc=fondo, ec="none"))
    ax.text(x + 1.3, y + 0.64, titulo, ha="center", va="center", color=texto, fontsize=12, fontweight="bold")
    ax.text(x + 1.3, y + 0.30, detalle, ha="center", va="center", color=texto, fontsize=9.5)


def flecha(x0, y0, x1, y1, color=MARINO, rad=0.0):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2, connectionstyle=f"arc3,rad={rad}"))


caja(0.4, 1.0, "jugador", "mira su casilla y elige", NARANJA)
caja(7.0, 1.0, "lago", "hielo, hoyos y meta", AZUL)
flecha(3.0, 1.75, 7.0, 1.75, rad=-0.25)
ax.text(5.0, 2.55, "se mueve: ← ↓ → ↑", ha="center", color=MARINO, fontsize=11)
flecha(7.0, 1.25, 3.0, 1.25, rad=-0.25)
ax.text(5.0, 0.2, "casilla nueva + puntos (1 si llegó a la meta, 0 si no)", ha="center", color=MARINO, fontsize=11)
ax.set_title("Una y otra vez, miles de veces: así aprende el jugador")
plt.show()

---
## Parte 1 — Conocer el lago (15 min)

> **En palabras simples (diapos 3 y 4).** El lago le dice al jugador en qué casilla está, el jugador elige uno de 4 movimientos y el lago le responde con la casilla nueva y unos puntos. Una **partida** dura hasta que el jugador cae a un hoyo o llega a la meta. Cada vuelta del bucle es **un instante de juego** (en los libros, un *paso*).

**Consigna**

1. Mira el lago y cómo se numeran sus casillas.
2. Mira qué devuelve el lago en cada instante.
3. Completa el ciclo de **una partida al azar** y mira jugar al jugador que se mueve sin pensar.

In [ ]:
lago = crear_lago("4x4")
fig, ax = plt.subplots(figsize=(4.2, 4.2))
dibujar_lago(lago, numeros=True, titulo="El lago de 4 × 4: 16 casillas")
plt.show()
print("El jugador parte arriba a la izquierda (casilla 0) y la meta está abajo a la derecha (casilla 15).")
print("Lo único que el jugador sabe es EN QUÉ CASILLA ESTÁ: un número de 0 a 15. No ve el mapa.")
display(pd.DataFrame({"movimiento": MOVIMIENTOS, "flecha": FLECHAS}, index=pd.Index(range(4), name="número")))

### Lo que devuelve cada instante

Cada vez que el jugador se mueve, `lago.step(movimiento)` devuelve 5 cosas: la **casilla nueva**, los **puntos** de ese instante, si la partida **terminó** (hoyo o meta), si se **cortó** por tiempo (100 instantes en el lago chico) y una información extra que aquí no usaremos.

In [ ]:
casilla, info = lago.reset(seed=SEED)
casilla_nueva, puntos, terminada, cortada, info = lago.step(2)   # movimiento 2: derecha
display(pd.DataFrame({"valor": [casilla, casilla_nueva, puntos, terminada, cortada],
                      "qué es": ["casilla donde estaba", "casilla nueva", "puntos de este instante",
                                 "¿terminó la partida?", "¿se cortó por tiempo?"]},
                     index=["antes", "casilla_nueva", "puntos", "terminada", "cortada"]))

### TODO 1 — Una partida al azar

Completa el ciclo: en cada instante, elige un movimiento al azar (`lago.action_space.sample()`), hazlo con `lago.step(mov)`, anota la casilla nueva en el camino y termina cuando la partida se acabe (`terminada or cortada`).

In [ ]:
# TODO 1: completa los ... (una partida con movimientos al azar)
casilla, info = lago.reset(seed=SEED)
lago.action_space.seed(SEED)
camino = [casilla]
puntos_total = 0
terminada = cortada = False
while not (terminada or cortada):
    mov = ...                                                     # un movimiento al azar
    casilla, puntos, terminada, cortada, info = ...               # hazlo
    camino.append(...)                                            # anota la casilla nueva
    puntos_total += puntos
print(f"Partida al azar: {len(camino) - 1} instantes, {puntos_total:.0f} puntos, terminó en la casilla {casilla}")

In [ ]:
# --- Autochequeo: la partida al azar ---
assert len(camino) > 1 and all(isinstance(int(c), int) for c in camino), \
    "Revisa que estés usando el movimiento en `lago.step(mov)` y que agregues `casilla` al camino en cada vuelta."
assert terminada or cortada, "El ciclo debe terminar con `terminada or cortada`."
print(f"✔ Partida al azar bien: {len(camino) - 1} instantes, terminó en la casilla {casilla}")
animar(lago, camino, titulo="Jugador al azar")

### El jugador al azar, 5.000 partidas

Con unas pocas partidas no se puede saber qué tan seguido llega a la meta: hay que jugar muchas.

In [ ]:
AZAR_4 = medir(lago, None, n=5000)
cinco = [jugar(lago, None, seed=s) for s in range(5)]
display(pd.DataFrame({"instantes": [p["instantes"] for p in cinco],
                      "¿llegó a la meta?": ["sí" if p["llego"] else "no (cayó a un hoyo)" for p in cinco]},
                     index=pd.Index(range(1, 6), name="partida")))
print(f"En 5.000 partidas al azar, el jugador llega a la meta en el {pct(AZAR_4)} de ellas.")

**Pregunta 1.** ¿Qué tan seguido llega a la meta el jugador que se mueve al azar? ¿Cómo terminan casi todas sus partidas, y en cuántos instantes?

**Respuesta:** (escribe aquí)

---
## Parte 2 — La tabla del jugador (25 min)

> **En palabras simples (diapo 9).** El jugador lleva un **cuaderno**: una tabla con una fila por casilla y una columna por movimiento. En cada cuadro anota **qué tan bueno resultó** hacer ese movimiento en esa casilla. Al principio todo está en cero. Cuando un movimiento lo lleva a la meta, sube ese número; y la próxima vez que llegue a la casilla de al lado, anota que también es buena, porque desde ahí se llega a una casilla buena. Así el buen número **se contagia hacia atrás**, desde la meta hasta el inicio. Esa es la **tabla Q** de la diapo 9.

Para jugar, el jugador mira su fila y elige el movimiento con el número más alto. A veces, por **curiosidad**, prueba uno al azar (lo veremos en la Parte 5).

Todo eso ya está escrito en las herramientas:

- `elegir(Q, casilla, curiosidad, rng)`: elige un movimiento (al azar por curiosidad, o el mejor de la tabla).
- `anotar(Q, casilla, mov, puntos, casilla_nueva, terminada, aprendizaje, paciencia)`: anota en la tabla lo que pasó.
  - `aprendizaje`: cuánto le cree a la última partida (0,8 = mucho).
  - `paciencia`: cuánto le importa lo que viene después (0,95 = bastante).

En los libros, estos valores se llaman *alfa*, *gamma* y *épsilon*. Esta es la tabla vacía:

In [ ]:
Q_vacia = np.zeros((16, 4))
display(pd.DataFrame(Q_vacia, columns=[f"{f} {m}" for f, m in zip(FLECHAS, MOVIMIENTOS)],
                     index=pd.Index(range(16), name="casilla")).head(6))
print(f"La tabla tiene {Q_vacia.shape[0]} filas (casillas) × {Q_vacia.shape[1]} columnas (movimientos) "
      f"= {Q_vacia.size} números. Arriba se ven las primeras 6 filas.")

### TODO 2 — Practicar: jugar y anotar

Completa el ciclo de una partida dentro de `practicar`. En cada instante:

1. **elige** un movimiento con `elegir(Q, casilla, cur, rng)`,
2. **hazlo** con `env.step(mov)`,
3. **anótalo** en la tabla con `anotar(Q, casilla, mov, puntos, casilla_nueva, terminada, aprendizaje, paciencia)`,
4. y **pasa** a la casilla nueva.

El resto de la función (repetir muchas partidas, bajar la curiosidad, guardar fotos de la tabla) ya está escrito.

In [ ]:
# TODO 2: completa los ... (una partida: elegir, hacer, anotar, pasar)

def practicar(env, n_partidas, aprendizaje=0.8, paciencia=0.95, curiosidad="baja", seed=SEED, fotos=()):
    """El jugador practica n_partidas y va anotando en su tabla Q.
    Devuelve la tabla, si llegó a la meta en cada partida y fotos de la tabla en las partidas pedidas."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))   # la tabla parte en cero
    llego = np.zeros(n_partidas)
    album = {}
    for partida in range(n_partidas):
        cur = curiosidad_que_baja(partida, n_partidas) if curiosidad == "baja" else curiosidad
        casilla, info = env.reset(seed=seed if partida == 0 else None)
        terminada = cortada = False
        while not (terminada or cortada):
            mov = ...                                                        # 1. elige un movimiento
            casilla_nueva, puntos, terminada, cortada, info = ...            # 2. hazlo
            ...                                                              # 3. anótalo en la tabla
            casilla = ...                                                    # 4. pasa a la casilla nueva
            llego[partida] += puntos
        if partida + 1 in fotos:
            album[partida + 1] = Q.copy()
    return Q, llego, album

Ahora el jugador practica **2.000 partidas** en el lago de 4 × 4 (tarda menos de un segundo).

In [ ]:
FOTOS = (63, 90, 100, 2000)   # la primera llegada a la meta (partida 63), el contagio y el final
t0 = time.time()
Q_4, llego_4, album_4 = practicar(lago, 2000, fotos=FOTOS)
EXITO_4 = medir(lago, Q_4)
print(f"2.000 partidas de práctica en {time.time() - t0:.1f} s. "
      f"Primera vez que llegó a la meta: partida {int(np.argmax(llego_4)) + 1}.")
print(f"Jugando con su tabla, ahora llega a la meta en el {EXITO_4:.0%} de las partidas (al azar: {pct(AZAR_4)}).")

In [ ]:
# --- Autochequeo: practicar ---
assert Q_4.shape == (16, 4), "La tabla debe tener 16 filas y 4 columnas."
assert Q_4.max() > 0, ("La tabla sigue en cero: revisa que llames a `anotar(...)` en cada instante "
                       "y que pases a la casilla nueva (`casilla = casilla_nueva`).")
assert EXITO_4 >= 0.99, ("El jugador todavía no cruza el lago. Revisa que el movimiento que eliges sea el que "
                         "le pasas a `env.step(mov)` y que anotes con la casilla de ANTES de moverte.")
print(f"✔ Tu jugador aprendió: con su tabla llega a la meta en el {EXITO_4:.0%} de las partidas")

### Cómo se llenó la tabla

Cada lago es una **foto de la tabla** en un momento de la práctica. Donde la tabla tiene un número mayor que cero, la casilla se pinta de azul (más fuerte = número más alto) y aparece la flecha del mejor movimiento. Donde todavía no sabe nada, hay un punto.

In [ ]:
fig, axes = plt.subplots(1, len(FOTOS), figsize=(4 * len(FOTOS), 4.3))
for ax, k in zip(axes, FOTOS):
    dibujar_lago(lago, Q=album_4[k], ax=ax, valores=True, titulo=f"después de {k:,} partidas".replace(",", "."))
plt.suptitle("El buen número aparece junto a la meta y se contagia hacia atrás", fontweight="bold", color=MARINO)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
ax.imshow(Q_4, cmap="Blues", aspect="auto")
for s in range(16):
    for a in range(4):
        ax.text(a, s, f"{Q_4[s, a]:.2f}".replace(".", ","), ha="center", va="center", fontsize=7.5,
                color="white" if Q_4[s, a] > 0.6 * Q_4.max() else MARINO)
ax.set(xticks=range(4), xticklabels=[f"{f} {m}" for f, m in zip(FLECHAS, MOVIMIENTOS)], yticks=range(16),
       ylabel="casilla", title="La tabla del jugador al final")
ax.grid(False)

ax = axes[1]
bloques = llego_4.reshape(-1, 100).mean(axis=1)
ax.plot(np.arange(1, len(bloques) + 1) * 100, bloques * 100, color=NARANJA, lw=2.5, marker="o", label="tu jugador practicando")
ax.axhline(AZAR_4 * 100, color=GRIS, ls="--", label="jugador al azar")
ax.set(xlabel="partidas de práctica", ylabel="% de partidas que llegan a la meta",
       title="Mientras practica, cada vez llega más seguido", ylim=(-3, 103))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
partida_4 = jugar(lago, Q_4)
print(f"Jugando con su tabla: {partida_4['instantes']} instantes, ¿llegó? {'sí' if partida_4['llego'] else 'no'}")
animar(lago, partida_4["camino"], titulo="Tu jugador, con su tabla")

**Pregunta 2.** ¿En qué casillas aparecieron primero los números de la tabla? ¿Por qué el buen número se va contagiando hacia atrás, desde la meta hasta el inicio? Mira la curva: ¿mejora de a poco o de golpe?

**Respuesta:** (escribe aquí)

---
## Parte 3 — El hielo resbala (15 min)

> **En palabras simples (diapo 5).** Ahora el hielo resbala: el jugador quiere ir a un lado, pero **solo 1 de cada 3 veces** va donde quiso; las otras veces se desliza hacia un costado. Lo que pase después sigue dependiendo solo de la casilla en que está (diapo 5), pero ya no es seguro. Por eso una sola partida engaña: a veces un buen movimiento termina en un hoyo por mala suerte. Conviene **aprender despacio** (no creerle tanto a la última partida) y **tener paciencia** (pensar en lo que viene después).

Comparamos dos formas de practicar, **repitiendo cada una con 5 semillas** (5 «suertes» distintas):

- (a) los valores de la Parte 2: `aprendizaje = 0.8`, `paciencia = 0.95`, 2.000 partidas;
- (b) aprender despacio y con paciencia: `aprendizaje = 0.1`, `paciencia = 0.99`, 20.000 partidas.

In [ ]:
lago_r = crear_lago("4x4", resbala=True)
MEJOR_4R = mejor_posible(lago_r)
t0 = time.time()
config = {"(a) aprendizaje 0,8 · paciencia 0,95 · 2.000 partidas": dict(n_partidas=2000, aprendizaje=0.8, paciencia=0.95),
          "(b) aprendizaje 0,1 · paciencia 0,99 · 20.000 partidas": dict(n_partidas=20000, aprendizaje=0.1, paciencia=0.99)}
resultados, tablas_r = {}, {}
for nombre, valores in config.items():
    exitos = []
    for s in range(N_SEMILLAS):
        Q, _, _ = practicar(lago_r, seed=s, **valores)
        exitos.append(medir(lago_r, Q))
        tablas_r.setdefault(nombre, Q)          # guardamos la tabla de la primera semilla
    resultados[nombre] = exitos
print(f"Listo en {time.time() - t0:.0f} s")
tabla_r = pd.DataFrame({"llega a la meta (promedio)": [f"{np.mean(v):.0%}" for v in resultados.values()],
                        "en cada semilla": [" · ".join(f"{x:.0%}" for x in v) for v in resultados.values()]},
                       index=list(resultados))
display(tabla_r)
print(f"El mejor jugador posible, que conoce el lago entero y cómo resbala, llega el {MEJOR_4R:.0%} de las veces.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4))
for ax, (nombre, Q) in zip(axes, tablas_r.items()):
    dibujar_lago(lago_r, Q=Q, ax=ax, titulo=nombre.split(" · ")[0] + f" · llega {medir(lago_r, Q):.0%}")
plt.suptitle("Su forma de jugar en el hielo resbaloso (semilla 0)", fontweight="bold", color=MARINO)
plt.tight_layout()
plt.show()
partida_r = jugar(lago_r, tablas_r[list(tablas_r)[1]], seed=3)
animar(lago_r, partida_r["camino"], titulo="(b) en hielo resbaloso")

**Pregunta 3.** ¿Por qué ni el mejor jugador posible llega siempre a la meta? ¿Cuál de las dos formas de practicar funcionó mejor, y por qué crees que en hielo resbaloso conviene aprender despacio? Mira las flechas de (b): ¿hay alguna que parezca rara (por ejemplo, que apunte contra el borde)? ¿Por qué podría convenir?

**Respuesta:** (escribe aquí)

---
## Parte 4 — El lago grande: 8 × 8 (20 min)

> **En palabras simples (diapos 8 y 9).** Ahora el lago tiene **64 casillas** y más hoyos, y la meta queda mucho más lejos. Al azar, el jugador casi nunca llega, así que tarda mucho más en encontrar la meta por primera vez; y la tabla crece: 64 filas en vez de 16. Primero en hielo firme y después, el desafío, en hielo resbaloso.

In [ ]:
grande = crear_lago("8x8")
grande_r = crear_lago("8x8", resbala=True)
AZAR_8 = medir(grande, None, n=5000)
fig, ax = plt.subplots(figsize=(5, 5))
dibujar_lago(grande, numeros=True, titulo="El lago de 8 × 8: 64 casillas")
plt.show()
print(f"Al azar, el jugador llega a la meta en el {pct(AZAR_8, 2)} de las partidas (en el lago chico: {pct(AZAR_4)}).")
print(f"La tabla ahora tiene 64 × 4 = {64 * 4} números (en el lago chico, {16 * 4}).")

### En hielo firme, con los valores de la Parte 2

Como el lago es más grande, le damos más práctica: 5.000 partidas.

In [ ]:
t0 = time.time()
Q_8, llego_8, album_8 = practicar(grande, 5000, fotos=(300, 1000, 5000))
print(f"5.000 partidas en {time.time() - t0:.1f} s · primera vez en la meta: partida {int(np.argmax(llego_8)) + 1} · "
      f"con su tabla llega el {medir(grande, Q_8):.0%} de las veces")
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
for ax, (k, Q) in zip(axes, album_8.items()):
    dibujar_lago(grande, Q=Q, ax=ax, valores=True, titulo=f"después de {k:,} partidas".replace(",", "."))
plt.tight_layout()
plt.show()
animar(grande, jugar(grande, Q_8)["camino"], titulo="Lago grande, hielo firme")

### TODO 3 — El desafío: el lago grande con hielo resbaloso

Con los valores de la Parte 2, el jugador casi no aprende en el lago grande resbaloso. **Elige tú** cuántas partidas practica, cuánto aprendizaje y cuánta paciencia, para que llegue a la meta **al menos 6 de cada 10 veces**. El mejor jugador posible llega unas 9 de cada 10.

Pistas: mira lo que funcionó en la Parte 3. Más de 20.000 partidas no hacen falta (cada 10.000 tardan unos segundos).

In [ ]:
# TODO 3: elige tus valores (reemplaza los ...)
MIS_VALORES = dict(n_partidas=..., aprendizaje=..., paciencia=...)

In [ ]:
t0 = time.time()
Q_8r, llego_8r, _ = practicar(grande_r, **MIS_VALORES)
EXITO_8R = medir(grande_r, Q_8r)
MEJOR_8R = mejor_posible(grande_r)
con_parte2 = medir(grande_r, practicar(grande_r, 5000)[0])
print(f"Con tus valores {MIS_VALORES}: practicó en {time.time() - t0:.0f} s y llega a la meta el {EXITO_8R:.0%} de las veces.")
print(f"Con los valores de la Parte 2 (5.000 partidas) llegaba el {con_parte2:.0%}; el mejor posible llega el {MEJOR_8R:.0%}.")

In [ ]:
# --- Autochequeo: el lago grande resbaloso ---
assert all(isinstance(v, (int, float)) for v in MIS_VALORES.values()), "Reemplaza los ... de MIS_VALORES por números."
assert EXITO_8R >= 0.6, (f"Tu jugador llega solo el {EXITO_8R:.0%} de las veces. Prueba aprender más despacio "
                         "(aprendizaje más chico), con más paciencia (cerca de 1) o con más partidas, como en la Parte 3.")
print(f"✔ ¡Cruzó el lago grande! Llega a la meta el {EXITO_8R:.0%} de las veces (el mejor posible: {MEJOR_8R:.0%})")

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
dibujar_lago(grande_r, Q=Q_8r, titulo=f"Su forma de jugar · llega {EXITO_8R:.0%}")
plt.show()
partida_8r = jugar(grande_r, Q_8r, seed=1)
print(f"Una partida: {partida_8r['instantes']} instantes, ¿llegó? {'sí' if partida_8r['llego'] else 'no'} "
      "(la animación muestra hasta 100 instantes)")
animar(grande_r, partida_8r["camino"], titulo="Lago grande resbaloso", max_cuadros=101)

**Pregunta 4.** ¿Qué valores elegiste y cuánto llegó tu jugador? ¿Por qué con los valores de la Parte 2 casi no aprendía? En el lago grande, ¿cuántos números tiene la tabla, y cuántos tendría si el lago fuera de 100 × 100?

**Respuesta:** (escribe aquí)

---
## Parte 5 — Explorar y diseñar los puntos (10 min)

> **En palabras simples (diapos 8 y 13).** Si el jugador siempre hace lo que su tabla dice que es mejor, nunca descubre caminos nuevos: se queda con el primero que encontró. Si a veces prueba algo al azar (por **curiosidad**), puede descubrir algo mejor… o caer a un hoyo. Y ojo con los puntos: el jugador aprende a hacer lo que le da puntos, no lo que nosotros queríamos.

### ¿Cuánta curiosidad?

En el lago grande resbaloso, con aprendizaje 0,1, paciencia 0,99 y 5.000 partidas, cambiamos solo la curiosidad: nunca prueba al azar (0), prueba 1 de cada 10 veces (0,1), o parte probando todo al azar y la va bajando de a poco (como hasta ahora).

In [ ]:
t0 = time.time()
curiosidades = {"nunca prueba al azar (0)": 0.0, "1 de cada 10 veces (0,1)": 0.1, "parte en 1 y va bajando": "baja"}
filas = {}
for nombre, cur in curiosidades.items():
    final, practicando = [], []
    for s in range(N_SEMILLAS):
        Q, llego, _ = practicar(grande_r, 5000, aprendizaje=0.1, paciencia=0.99, curiosidad=cur, seed=s)
        final.append(medir(grande_r, Q))
        practicando.append(llego.mean())
    filas[nombre] = {"llega al final, con su tabla": f"{np.mean(final):.0%}",
                     "en cada semilla": " · ".join(f"{x:.0%}" for x in final),
                     "llegaba mientras practicaba": f"{np.mean(practicando):.0%}"}
print(f"Listo en {time.time() - t0:.0f} s")
display(pd.DataFrame(filas).T)

### ¿Y si cambiamos los puntos?

Ahora le damos **1 punto por cada instante en que no cae a un hoyo** (y nada especial por llegar a la meta). Suena razonable: «premiemos que no se caiga». Mira qué aprende, en el lago chico de hielo firme.

In [ ]:
class PuntoPorInstante(gym.Wrapper):
    """Puntos distintos: 1 por cada instante en que el jugador no cae a un hoyo (0 si cae)."""
    def step(self, mov):
        casilla, puntos, terminada, cortada, info = self.env.step(mov)
        cayo = terminada and mapa(self.env).flatten()[casilla] == "H"
        return casilla, 0.0 if cayo else 1.0, terminada, cortada, info


lago_trampa = PuntoPorInstante(crear_lago("4x4"))
Q_t, _, _ = practicar(lago_trampa, 2000)
partida_t = jugar(lago_trampa, Q_t)
print(f"Con 1 punto por instante: jugó {partida_t['instantes']} instantes, ganó {partida_t['puntos']:.0f} puntos "
      f"y ¿llegó a la meta? {'sí' if partida_t['llego'] else 'no'}")
fig, ax = plt.subplots(figsize=(4.2, 4.2))
dibujar_lago(lago_trampa, Q=Q_t, titulo="Su forma de jugar con 1 punto por instante")
plt.show()
animar(lago_trampa, partida_t["camino"], titulo="1 punto por instante")

**Pregunta 5.** ¿Qué pasó con el jugador que nunca prueba al azar? ¿Por qué conviene partir con mucha curiosidad e ir bajándola? Y con 1 punto por instante, ¿qué aprendió a hacer el jugador? ¿Es lo que queríamos?

**Respuesta:** (escribe aquí)

---
## Parte 6 — Cuando la tabla no alcanza: Mario (5 min)

> **En palabras simples (diapo 10).** En el lago el jugador solo necesita saber en qué casilla está: 16 o 64 posibilidades, y la tabla cabe en una pantalla. En un videojuego como Mario, lo que el jugador ve es **la pantalla completa**, y hay tantas pantallas distintas que ninguna tabla alcanzaría: casi nunca vería dos veces la misma. Por eso, en vez de una tabla, una **red convolucional** (como la de la Clase 11) mira la pantalla y decide qué botón apretar. La idea es la misma: puntos por avanzar, probar y repetir lo que funcionó.

Este es el **Mario del profe**: una red que practicó 2 millones de instantes de juego (unos 51 minutos en una GPU). El video muestra cómo jugaba después de distintas cantidades de práctica.

In [ ]:
from pathlib import Path
import urllib.request

VIDEO = Path("mario_progreso.mp4")
URL_VIDEO = "https://github.com/diegoalrv/ml-ein143a25/releases/download/lab13-mario/mario_progreso.mp4"
try:
    if not VIDEO.exists():
        urllib.request.urlretrieve(URL_VIDEO, VIDEO)
    display(Video(str(VIDEO), embed=True, width=512))
except Exception as e:  # sin internet: el video se ve igual en el navegador
    print(f"No pude bajar el video ({e}). Míralo aquí: {URL_VIDEO}")

> **Para la casa (opcional).** En `Lab13_Mario_estudiante.ipynb` (en esta misma carpeta) entrenas tu propio Mario durante 15 minutos y lo comparas con el del profe. Pide Python 3.13 en tu computador.

**Pregunta 6.** Si quisiéramos usar una tabla para Mario, ¿qué serían las filas? ¿Por qué no alcanza?

**Respuesta:** (escribe aquí)

---
## Parte 7 — Reflexión (5 min)

**Pregunta 7.1.** ¿En qué se diferencia cómo aprendió el jugador del lago de cómo aprendió la CNN de la Clase 11 a reconocer ropa?

**Respuesta:** (escribe aquí)

**Pregunta 7.2.** En una frase: ¿qué guarda la tabla del jugador y cómo la usa para decidir?

**Respuesta:** (escribe aquí)

**Pregunta 7.3.** ¿Dónde más se podría usar esta idea de aprender probando y recibiendo puntos?

**Respuesta:** (escribe aquí)

---
## Entregable

Sube a AULA-USM este notebook (`Lab13_FrozenLake_estudiante.ipynb`) **ejecutado**, con:

1. Las respuestas de las preguntas 1 a 7.
2. **Tus valores del lago grande** (TODO 3) y qué tan seguido llegó tu jugador: lo imprime la celda de abajo.

In [ ]:
print(f"Mis valores del lago grande: {MIS_VALORES} → llega a la meta el {EXITO_8R:.0%} de las veces "
      f"(el mejor posible: {MEJOR_8R:.0%})")
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")